# *Libra* - Extending Model Functionality with Mixins
> Brady Spears, Los Alamos National Laboratory

---

## About *Libra*
`Libra` is an open-source Python package for managing object-relation-mapped 
(ORM) instances (A.K.A. "Models") across various schemas and various relational 
database backends. `Libra` was developed by Brady Spears with 
contirubtions from Christine Gammans, Jonathan MacCarthy, Richard Alfaro-Diaz,
and Jeremy Webster at Los Alamos National Laboratory.

---

## About this Notebook
This notebook demonstrates how models belonging to a `Schema` can optionally 
inherit functionality through built-in or custom 'Mix-in' classes. `Libra` comes
pre-packaged with two: `FlatFileMixin` and `QCMixin`, which act to extend the 
'\_\_new\_\_()' method of inheriting models. 

---

### `FlatFileMixin` - Supports the reading & writing of flat files

Flat files are strictly-formatted text files containing rows of data. It is 
often desirable to have tabular data structured in these flat files because they
are easy to ingest into an SQL database. The `FlatFileMixin` class extends 
support to a `Libra`-based model, so that it too can successfully parse a flat
file into a list of models or write out a list of models to a flat file. By 
default, all models belonging to a `Schema` come pre-loaded with the 
`FlatFileMixin` class and its added functionality.

Note: It is important to consider that these Mix-in classes are often dependent 
on additional information at the column level. To achieve this, it might be 
necessary to pass specific key-value pairs to each Column object's 'info' 
dictionary during declaration of the model.

Consider the following example showing the reading of a flat file into a 
series of models:

In [52]:
from datetime import datetime, timezone

from sqlalchemy import Column
from sqlalchemy import DateTime, Integer, String

from libra import Schema

css = Schema('CSS 3.0')

# Define the model in-line with 'info' dictionary containing 'format' key
@css.add_model
class network:
    net     = Column(String(8), default = '-', info = {'format' : '8.8s'})
    netname = Column(String(15), default = '-', info = {'format' : '15.15s'})
    nettype = Column(String(4), default = '-', info = {'format' : '4.4s'})
    auth    = Column(String(15), default = '-', info = {'format' : '15.15s'})
    commid  = Column(Integer, default = -1, info = {'format' : '8d'})
    lddate  = Column(DateTime, nullable = False, onupdate = datetime.now(timezone.utc),
                default = datetime.now(timezone.utc), info = {'format' : '%Y-%m-%d %H:%M:%S'})

    pk = ['net']

class Network(css.network):
    __tablename__ = 'mixin_network'

# Read in a CSS3.0 Network flat file
with open('data/css.network', 'r') as f:
    lines = f.readlines()

print(*[line for line in lines])

USGS     USGS Stations   ww   USGS                  -1 2003-11-14 10:54:26\n
 TX       TX Seismic Net  lo   FDSN                  -1 2023-01-30 14:00:26\n
 FA       UCLA Seismology lo   FDSN                  -1 2014-03-20 11:06:42\n


Now that we have a model with specific formatting for each Column defined, we 
can use one of `Libra's` built-in ease-of-use functions to push the flatfile 
lines to a list of models:

In [53]:
from libra.func import from_string

# Convert lines to a list of Network objects
nets = from_string(Network(), lines)

print(*[net for net in nets], sep='\n')

Network(net='USGS', netname='USGS Stations', nettype='ww', auth='USGS', commid=-1, lddate=2003-11-14 10:54:26)
Network(net='TX', netname='TX Seismic Net', nettype='lo', auth='FDSN', commid=-1, lddate=2023-01-30 14:00:26)
Network(net='FA', netname='UCLA Seismology', nettype='lo', auth='FDSN', commid=-1, lddate=2014-03-20 11:06:42)


It would also be nice to go the other way - ORM to flat file:

In [58]:
from libra.func import from_string, to_string

nets = [
    Network('USGS', 'USGS Stations', 'ww', 'USGS', -1, datetime(2003, 11, 14, 10, 54, 26)),
    Network('TX', 'TX Seismic Net', 'lo', 'FDSN', -1, datetime(2023, 1, 30, 14, 0, 26)),
    Network('FA', 'UCLA Seismology', 'lo', 'FDSN', -1, datetime(2014, 3, 20, 11, 6, 42))
]

string = to_string(nets)

print(*[s for s in string])

USGS     USGS Stations   ww   USGS                  -1 2003-11-14 10:54:26
 TX       TX Seismic Net  lo   FDSN                  -1 2023-01-30 14:00:26
 FA       UCLA Seismology lo   FDSN                  -1 2014-03-20 11:06:42

